In [1]:
!export HF_ENDPOINT=https://hf-mirror.com

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from scipy.stats import loguniform

df = pd.read_csv('../dataset/dataset_engineered.csv')

features = [
    'followers', 'duration', 'musicOriginal', 'hour', 'weekday',
    'hist_median_views', 'hist_p70_views', 'hist_p90_views', 
    'hist_like_rate', 'hist_comment_rate', 'hist_share_rate',
    'n_hashtags', 'has_fyp', 'has_viral', 'has_foryou', 
    'caption_len', 'has_emoji', 'has_question', 'has_exclamation',
    'viral_potential', 'engagement_total_hist', 'is_peak_hour',
    'follower_tier', 'views_efficiency_trend'
]

df['is_viral'] = (df['followers'] * df['explosion_score'] > df['hist_p70_views']).astype(int)

target = 'is_viral'
print(f"Percentage (Viral): {df['is_viral'].mean():.2%}")

train_val_df = df[df['video_rank'].between(11, 28)].copy()
test_df = df[df['video_rank'].isin([29, 30])].copy()

X_train_val = train_val_df[features]
y_train_val = train_val_df[target]
X_test = test_df[features]
y_test = test_df[target]

test_fold = np.where(train_val_df['video_rank'] <= 26, -1, 0)
ps = PredefinedSplit(test_fold)

scaler = StandardScaler()
X_train_val_scaled = scaler.fit_transform(X_train_val)
X_test_scaled = scaler.transform(X_test)

def evaluate_binary_classifier(name, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
    cm = confusion_matrix(y_true, y_pred)
    
    print(f"\n[{name}]")
    print(f"Accuracy: {acc:.4f} | F1-Macro: {f1:.4f} | AUC-ROC(OvR): {auc:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print("-" * 40)

# A. Linear Regression
lr_search = RandomizedSearchCV(
    LogisticRegression(max_iter=1000), 
    {'C': loguniform(1e-4, 1e2), 'class_weight': ['balanced', None]}, 
    n_iter=30, cv=ps, scoring='f1', random_state=42, n_jobs=-1
).fit(X_train_val_scaled, y_train_val)
evaluate_binary_classifier("Logistic Regression", y_test, 
                           lr_search.predict(X_test_scaled), 
                           lr_search.predict_proba(X_test_scaled)[:, 1])

# B. KNN
knn_search = RandomizedSearchCV(
    KNeighborsClassifier(), 
    {'n_neighbors': np.arange(1, 51), 'weights': ['uniform', 'distance']}, 
    n_iter=30, cv=ps, scoring='f1', random_state=42, n_jobs=-1
).fit(X_train_val_scaled, y_train_val)
evaluate_binary_classifier("KNN Classifier", y_test, 
                           knn_search.predict(X_test_scaled), 
                           knn_search.predict_proba(X_test_scaled)[:, 1])

# C. Random Forest
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42), 
    {'n_estimators': [100, 200, 500], 'max_depth': [10, 20, None], 'class_weight': ['balanced', 'balanced_subsample', None]}, 
    n_iter=20, cv=ps, scoring='f1', random_state=42, n_jobs=-1
).fit(X_train_val_scaled, y_train_val)
evaluate_binary_classifier("Random Forest Classifier", y_test, 
                           rf_search.predict(X_test_scaled), 
                           rf_search.predict_proba(X_test_scaled)[:, 1])

Percentage (Viral): 29.08%

[Logistic Regression]
Accuracy: 0.4978 | F1-Macro: 0.4708 | AUC-ROC(OvR): 0.6062
Confusion Matrix:
[[327 404]
 [ 50 123]]
----------------------------------------

[KNN Classifier]
Accuracy: 0.6350 | F1-Macro: 0.5256 | AUC-ROC(OvR): 0.5470
Confusion Matrix:
[[504 227]
 [103  70]]
----------------------------------------

[Random Forest Classifier]
Accuracy: 0.6615 | F1-Macro: 0.5589 | AUC-ROC(OvR): 0.6310
Confusion Matrix:
[[517 214]
 [ 92  81]]
----------------------------------------


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from scipy.stats import loguniform

df = pd.read_csv('../dataset/dataset_engineered.csv')

features = [
    'followers', 'duration', 'musicOriginal', 'hour', 'weekday',
    'hist_median_views', 'hist_p70_views', 'hist_p90_views', 
    'hist_like_rate', 'hist_comment_rate', 'hist_share_rate',
    'n_hashtags', 'has_fyp', 'has_viral', 'has_foryou', 
    'caption_len', 'has_emoji', 'has_question', 'has_exclamation',
    'viral_potential', 'engagement_total_hist', 'is_peak_hour',
    'follower_tier', 'views_efficiency_trend'
]

actual_views = df['followers'] * df['explosion_score']

df['viral_level'] = 0
df.loc[(actual_views >= df['hist_median_views']) & (actual_views <= df['hist_p70_views']), 'viral_level'] = 1
df.loc[actual_views > df['hist_p70_views'], 'viral_level'] = 2

target = 'viral_level'
print(f"Distribution：\n{df['viral_level'].value_counts(normalize=True).sort_index()}")

train_val_df = df[df['video_rank'].between(11, 28)].copy()
test_df = df[df['video_rank'].isin([29, 30])].copy()

X_train_val = train_val_df[features]
y_train_val = train_val_df[target]
X_test = test_df[features]
y_test = test_df[target]

test_fold = np.where(train_val_df['video_rank'] <= 26, -1, 0)
ps = PredefinedSplit(test_fold)

scaler = StandardScaler()
X_train_val_scaled = scaler.fit_transform(X_train_val)
X_test_scaled = scaler.transform(X_test)

def evaluate_multi_classifier(name, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')
    auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
    cm = confusion_matrix(y_true, y_pred)
    
    print(f"\n[{name}]")
    print(f"Accuracy: {acc:.4f} | F1-Macro: {f1:.4f} | AUC-ROC(OvR): {auc:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    print("-" * 40)

# A. Logistic Regression
lr_search = RandomizedSearchCV(
    LogisticRegression(max_iter=1000), 
    {'C': loguniform(1e-4, 1e2), 'class_weight': ['balanced']},
    n_iter=20, cv=ps, scoring='f1_macro', random_state=42, n_jobs=-1
).fit(X_train_val_scaled, y_train_val)
evaluate_multi_classifier("Logistic Regression", y_test, lr_search.predict(X_test_scaled), lr_search.predict_proba(X_test_scaled))

# B. KNN
knn_search = RandomizedSearchCV(
    KNeighborsClassifier(), 
    {'n_neighbors': np.arange(1, 51), 'weights': ['uniform', 'distance']}, 
    n_iter=20, cv=ps, scoring='f1_macro', random_state=42, n_jobs=-1
).fit(X_train_val_scaled, y_train_val)
evaluate_multi_classifier("KNN Classifier", y_test, knn_search.predict(X_test_scaled), knn_search.predict_proba(X_test_scaled))

# C. Random Forest
rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42), 
    {
        'n_estimators': [100, 200], 
        'max_depth': [10, 20, None], 
        'class_weight': ['balanced', 'balanced_subsample', None]
    }, 
    n_iter=15, cv=ps, scoring='f1_macro', random_state=42, n_jobs=-1
).fit(X_train_val_scaled, y_train_val)
evaluate_multi_classifier("Random Forest Classifier", y_test, rf_search.predict(X_test_scaled), rf_search.predict_proba(X_test_scaled))

Distribution：
0    0.555248
1    0.153965
2    0.290786
Name: viral_level, dtype: float64

[Logistic Regression]
Accuracy: 0.4126 | F1-Macro: 0.3548 | AUC-ROC(OvR): 0.5749
Confusion Matrix:
[[235  96 300]
 [ 36  24  40]
 [ 43  16 114]]
----------------------------------------

[KNN Classifier]
Accuracy: 0.6250 | F1-Macro: 0.3764 | AUC-ROC(OvR): 0.5515
Confusion Matrix:
[[507  10 114]
 [ 73   4  23]
 [116   3  54]]
----------------------------------------

[Random Forest Classifier]
Accuracy: 0.5066 | F1-Macro: 0.4241 | AUC-ROC(OvR): 0.6292
Confusion Matrix:
[[333 101 197]
 [ 46  37  17]
 [ 56  29  88]]
----------------------------------------


##### TEST: MLP

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from torch.utils.data import DataLoader, TensorDataset

df = pd.read_csv('dataset_engineered.csv')

features = [
    'followers', 'duration', 'musicOriginal', 'hour', 'weekday',
    'hist_median_views', 'hist_p70_views', 'hist_p90_views', 
    'hist_like_rate', 'hist_comment_rate', 'hist_share_rate',
    'n_hashtags', 'has_fyp', 'has_viral', 'has_foryou', 
    'caption_len', 'has_emoji', 'has_question', 'has_exclamation',
    'viral_potential', 'engagement_total_hist', 'is_peak_hour',
    'follower_tier', 'views_efficiency_trend'
]

actual_views = df['followers'] * df['explosion_score']
df['label_binary'] = (actual_views > df['hist_p70_views']).astype(int)
df['label_triple'] = 0
df.loc[(actual_views >= df['hist_median_views']) & (actual_views <= df['hist_p70_views']), 'label_triple'] = 1
df.loc[actual_views > df['hist_p70_views'], 'label_triple'] = 2

class TikTokNumericalMLP(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        return self.network(x)

def run_pure_num_experiment(num_classes, label_col):
    task_name = "Binary classification" if num_classes == 2 else "Three classification"
    print(f"\n" + "="*20 + f" MLP {task_name} " + "="*20)

    train_idx = df[df['video_rank'].between(11, 28)].index
    test_idx = df[df['video_rank'].isin([29, 30])].index

    scaler = StandardScaler()
    X_train = scaler.fit_transform(df.loc[train_idx, features])
    X_test = scaler.transform(df.loc[test_idx, features])
    
    y_train = torch.LongTensor(df.loc[train_idx, label_col].values)
    y_test = df.loc[test_idx, label_col].values

    train_ds = TensorDataset(torch.FloatTensor(X_train), y_train)
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TikTokNumericalMLP(len(features), num_classes=num_classes).to(device)
    
    criterion = nn.CrossEntropyLoss() 
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

    model.train()
    for epoch in range(80):
        for bx, by in train_loader:
            optimizer.zero_grad()
            logits = model(bx.to(device))
            loss = criterion(logits, by.to(device))
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(torch.FloatTensor(X_test).to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='macro')
    if num_classes == 2:
        auc = roc_auc_score(y_test, probs[:, 1])
    else:
        auc = roc_auc_score(y_test, probs, multi_class='ovr', average='macro')

    print(f"Accuracy: {acc:.4f} | F1-Macro: {f1:.4f} | AUC-ROC: {auc:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, preds))

run_pure_num_experiment(num_classes=2, label_col='label_binary')
run_pure_num_experiment(num_classes=3, label_col='label_triple')


==================== MLP Binary classification ====================
Accuracy: 0.8026 | F1-Macro: 0.4505 | AUC-ROC: 0.5904
Confusion Matrix:
[[747   5]
 [179   1]]

==================== MLP Three classification ====================
Accuracy: 0.6867 | F1-Macro: 0.3314 | AUC-ROC: 0.5618
Confusion Matrix:
[[623   6  20]
 [ 97   3   3]
 [166   0  14]]
